In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import triplet_lineage as tl


In [ ]:
import cassiopeia as cas
import cProfile
from collections import defaultdict
import copy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import time
from tqdm.auto import tqdm


In [ ]:
k_cand = 100
lamb = 0.5
num_states = 5
q_dist = dict(zip(range(1, num_states + 1), [1 / num_states] * num_states))
depth = 8
d_star = 0.9
l = 1 / 9
dists = []
sims1 = []
wrong1 = 0
num_simulations = 100

algorithms = {
    "Vanilla Greedy": cas.solver.VanillaGreedySolver(),
    "UPGMA": cas.solver.UPGMASolver(dissimilarity_function=cas.solver.dissimilarity.weighted_hamming_distance),
    "NeighborJoining": cas.solver.NeighborJoiningSolver(
        dissimilarity_function=cas.solver.dissimilarity.weighted_hamming_distance, add_root=True
    ),
}
algorithm_to_performance_triplets_without_missing_data = defaultdict(list)
algorithm_to_performance_rf_without_missing_data=defaultdict(list)

for _ in tqdm(range(num_simulations)):
    ground_truth_tree = tl.complete_binary_tree_sim(k_cand, q_dist, lamb, depth)
    triplets = tl.find_recon_triplets(ground_truth_tree)
    recon_tree1 = tl.build_tree_from_triplet_partition(ground_truth_tree, triplets)
    accuracy_tree1 = tl.calculate_triplets_correct(ground_truth_tree, recon_tree1)
    algorithm_to_performance_triplets_without_missing_data["MAX-Cut"].append(accuracy_tree1)
    rf, rf_normailzed =tl.calculate_rf_for_maxcut(ground_truth_tree,recon_tree1)
    algorithm_to_performance_rf_without_missing_data["MAX-Cut"].append(rf_normailzed)

    for algorithm_name in tqdm(algorithms.keys()):
        algorithm = algorithms[algorithm_name]
        
        reconstructed_tree = cas.data.CassiopeiaTree(character_matrix = ground_truth_tree.character_matrix, missing_state_indicator = -1)
        algorithm.solve(reconstructed_tree)
        
        # ground_truth_tree.collapse_mutationless_edges(infer_ancestral_characters = False)
        reconstructed_tree.collapse_mutationless_edges(infer_ancestral_characters = True)

        rf, rf_max = cas.critique.compare.robinson_foulds(ground_truth_tree, reconstructed_tree)
        
        triplets = cas.critique.compare.triplets_correct(ground_truth_tree, reconstructed_tree, number_of_trials=500)
        algorithm_to_performance_triplets_without_missing_data[algorithm_name].append(np.mean(list(triplets[0].values())))
        
        algorithm_to_performance_rf_without_missing_data[algorithm_name].append(rf / rf_max)
        

csv

In [ ]:
def save_results_to_csv(algorithm_to_performance_rf, algorithm_to_performance_triplets, filename_rf, filename_triplets):
    rf_data = []
    triplets_data = []

    for algorithm, scores in algorithm_to_performance_rf.items():
        for score in scores:
            rf_data.append({"Algorithm": algorithm, "RF_Score": score})
    rf_df = pd.DataFrame(rf_data)
    rf_df.to_csv(filename_rf, index=False)

    for algorithm, scores in algorithm_to_performance_triplets.items():
        for score in scores:
            triplets_data.append({"Algorithm": algorithm, "Triplet_Score": score})
    triplets_df = pd.DataFrame(triplets_data)
    triplets_df.to_csv(filename_triplets, index=False)

save_results_to_csv(
    algorithm_to_performance_rf_without_missing_data,
    algorithm_to_performance_triplets_without_missing_data,
    PROJECT_ROOT / "results" / "rf_scores_without_missing.csv",
    PROJECT_ROOT / "results" / "triplet_scores_without_missing.csv",
)

In [ ]:
k_cand = 100
lamb = 0.5
num_states = 5
q_dist = dict(zip(range(1, num_states + 1), [1 / num_states] * num_states))
depth = 8
d_star = 0.9
l = 1 / 9
dists = []
sims1 = []
wrong1 = 0

num_simulations = 100
algorithms = {
    "Vanilla Greedy": cas.solver.VanillaGreedySolver(),
    "UPGMA": cas.solver.UPGMASolver(dissimilarity_function=cas.solver.dissimilarity.weighted_hamming_distance),
    "NeighborJoining": cas.solver.NeighborJoiningSolver(
        dissimilarity_function=cas.solver.dissimilarity.weighted_hamming_distance, add_root=True
    ),
}
algorithm_to_performance_triplets_with_missing_data = defaultdict(list)
algorithm_to_performance_rf_with_missing_data=defaultdict(list)

for _ in tqdm(range(num_simulations)):
    ground_truth_tree = tl.complete_binary_missing_tree_sim(k_cand, q_dist, lamb, depth)
    triplets = tl.find_recon_triplets(ground_truth_tree)
    recon_tree1 = tl.build_tree_from_triplet_partition(ground_truth_tree, triplets)
    accuracy_tree1 = tl.calculate_triplets_correct(ground_truth_tree, recon_tree1)
    algorithm_to_performance_triplets_with_missing_data["MAX-Cut"].append(accuracy_tree1)
    rf, rf_normailzed =tl.calculate_rf_for_maxcut(ground_truth_tree,recon_tree1)
    algorithm_to_performance_rf_with_missing_data["MAX-Cut"].append(rf_normailzed)

    for algorithm_name in tqdm(algorithms.keys()):
        algorithm = algorithms[algorithm_name]
        
        reconstructed_tree = cas.data.CassiopeiaTree(character_matrix = ground_truth_tree.character_matrix, missing_state_indicator = -1)
        algorithm.solve(reconstructed_tree)
        
        # ground_truth_tree.collapse_mutationless_edges(infer_ancestral_characters = False)
        reconstructed_tree.collapse_mutationless_edges(infer_ancestral_characters = True)

        rf, rf_max = cas.critique.compare.robinson_foulds(ground_truth_tree, reconstructed_tree)
        
        triplets = cas.critique.compare.triplets_correct(ground_truth_tree, reconstructed_tree, number_of_trials=500)
        algorithm_to_performance_triplets_with_missing_data[algorithm_name].append(np.mean(list(triplets[0].values())))
        
        algorithm_to_performance_rf_with_missing_data[algorithm_name].append(rf / rf_max)
        

In [ ]:
save_results_to_csv(
    algorithm_to_performance_rf_with_missing_data,
    algorithm_to_performance_triplets_with_missing_data,
    PROJECT_ROOT / "results" / "rf_scores_with_missing.csv",
    PROJECT_ROOT / "results" / "triplet_scores_with_missing.csv",
)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection

rf_df1 = pd.read_csv(PROJECT_ROOT / "results" / "rf_scores_without_missing.csv")
triplets_df1 = pd.read_csv(PROJECT_ROOT / "results" / "triplet_scores_without_missing.csv")
rf_df2 = pd.read_csv(PROJECT_ROOT / "results" / "rf_scores_with_missing.csv")
triplets_df2 = pd.read_csv(PROJECT_ROOT / "results" / "triplet_scores_with_missing.csv")

custom_colors = ["#B883D4", "#96C37D", "#9998FF", "#F1D77E"]

rf_df1["1 - RF_Score"] = 1 - rf_df1["RF_Score"]
rf_df2["1 - RF_Score"] = 1 - rf_df2["RF_Score"]

def set_violin_outline_only(ax, palette):
    for i, collection in enumerate(ax.collections):
        if isinstance(collection, PolyCollection):
            collection.set_facecolor('none')
            collection.set_edgecolor(palette[i % len(palette)])
            collection.set_linewidth(2)

# =================== Triplet Score ===================
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes = axes.flatten()

# (a)
sns.violinplot(ax=axes[0], y="Triplet_Score", x="Algorithm", hue="Algorithm",
               data=triplets_df1, palette=custom_colors, inner="box", legend=False)
set_violin_outline_only(axes[0], custom_colors)
axes[0].axhline(y=0.64, color='red', linestyle='--', linewidth=1)
axes[0].text(x=len(axes[0].get_xticks()) - 1.5, y=0.64, s="Low bound", 
             color='red', fontsize=10, va='bottom', ha='right')
axes[0].set_ylabel("Triplet Score")
axes[0].set_xlabel("")
axes[0].text(-0.8, axes[0].get_ylim()[1], "A", fontsize=14, weight='bold')
plt.setp(axes[0].get_xticklabels(), rotation=30, ha="right")

# (b)
sns.violinplot(ax=axes[1], y="Triplet_Score", x="Algorithm", hue="Algorithm",
               data=triplets_df2, palette=custom_colors, inner="box", legend=False)
set_violin_outline_only(axes[1], custom_colors)
axes[1].axhline(y=0.64, color='red', linestyle='--', linewidth=1)
axes[1].text(x=len(axes[1].get_xticks()) - 1.5, y=0.64, s="Low bound", 
             color='red', fontsize=10, va='bottom', ha='right')
axes[1].set_ylabel("Triplet Score")
axes[1].set_xlabel("")
# axes[1].text(-0.8, axes[1].get_ylim()[1], "b", fontsize=12, weight='bold')
plt.setp(axes[1].get_xticklabels(), rotation=30, ha="right")

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "triplets-violin-plots.png")
plt.show()

# =================== 1 - RF Score ===================
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes = axes.flatten()

# (c)
sns.violinplot(ax=axes[0], y="1 - RF_Score", x="Algorithm", hue="Algorithm",
               data=rf_df1, palette=custom_colors, inner="box", legend=False)
set_violin_outline_only(axes[0], custom_colors)
axes[0].set_ylabel("1 - RF Score")
axes[0].set_xlabel("")
axes[0].text(-0.8, axes[0].get_ylim()[1], "B", fontsize=14, weight='bold')
plt.setp(axes[0].get_xticklabels(), rotation=30, ha="right")

# (d)
sns.violinplot(ax=axes[1], y="1 - RF_Score", x="Algorithm", hue="Algorithm",
               data=rf_df2, palette=custom_colors, inner="box", legend=False)
set_violin_outline_only(axes[1], custom_colors)
axes[1].set_ylabel("1 - RF Score")
axes[1].set_xlabel("")
# axes[1].text(-0.8, axes[1].get_ylim()[1], "d", fontsize=12, weight='bold')
plt.setp(axes[1].get_xticklabels(), rotation=30, ha="right")

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "rf-violin-plots.png")
plt.show()

'''
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

rf_df1 = pd.read_csv(PROJECT_ROOT / "results" / "rf_scores_without_missing.csv")
triplets_df1 = pd.read_csv(PROJECT_ROOT / "results" / "triplet_scores_without_missing.csv")
rf_df2 = pd.read_csv(PROJECT_ROOT / "results" / "rf_scores_with_missing.csv")
triplets_df2 = pd.read_csv(PROJECT_ROOT / "results" / "triplet_scores_with_missing.csv")

custom_colors = ["#026EC2", "#EC7B2F", "#FFDF69", "#84c3b7"]

rf_df1["1 - RF_Score"] = 1 - rf_df1["RF_Score"]
rf_df2["1 - RF_Score"] = 1 - rf_df2["RF_Score"]

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes = axes.flatten()

# (a) Triplet Score Without Missing Data
sns.violinplot(ax=axes[0], y="Triplet_Score", x="Algorithm", hue="Algorithm",
               data=triplets_df1, palette=custom_colors, inner="box", legend=False)
axes[0].axhline(y=0.64, color='red', linestyle='--', linewidth=1, label='y = 0.64')
axes[0].text(x=len(axes[0].get_xticks()) - 1.5, y=0.64, s="Low bound", 
             color='red', fontsize=10, va='bottom', ha='right')
axes[0].set_ylabel("Triplet Score")
axes[0].set_xlabel("")
axes[0].text(-0.8, axes[0].get_ylim()[1], "a", fontsize=12, weight='bold')
plt.setp(axes[0].get_xticklabels(), rotation=30, ha="right")

# (d) Triplet Score With Missing Data
sns.violinplot(ax=axes[1], y="Triplet_Score", x="Algorithm", hue="Algorithm",
               data=triplets_df2, palette=custom_colors, inner="box", legend=False)
axes[1].axhline(y=0.64, color='red', linestyle='--', linewidth=1)
axes[1].text(x=len(axes[1].get_xticks()) - 1.5, y=0.64, s="Low bound", 
             color='red', fontsize=10, va='bottom', ha='right')
axes[1].set_ylabel("Triplet Score")
axes[1].set_xlabel("")
axes[1].text(-0.8, axes[1].get_ylim()[1], "b", fontsize=12, weight='bold')
plt.setp(axes[1].get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "triplets-violin-plots.png")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes = axes.flatten()
# (c) 1 - RF Score Without Missing Data
sns.violinplot(ax=axes[0], y="1 - RF_Score", x="Algorithm", hue="Algorithm",
               data=rf_df1, palette=custom_colors, inner="box", legend=False)
axes[0].set_ylabel("1 - RF Score")
axes[0].set_xlabel("")
axes[0].text(-0.8, axes[0].get_ylim()[1], "c", fontsize=12, weight='bold')
plt.setp(axes[0].get_xticklabels(), rotation=30, ha="right")

# (d) 1 - RF Score With Missing Data
sns.violinplot(ax=axes[1], y="1 - RF_Score", x="Algorithm", hue="Algorithm",
               data=rf_df2, palette=custom_colors, inner="box", legend=False)
axes[1].set_ylabel("1 - RF Score")
axes[1].set_xlabel("")
axes[1].text(-0.8, axes[1].get_ylim()[1], "d", fontsize=12, weight='bold')
plt.setp(axes[1].get_xticklabels(), rotation=30, ha="right")

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "rf-violin-plots.png")
plt.show()
'''

In [ ]:
import math
import numpy as np
from scipy.stats import norm
n = 256 * 256 * 256 / (3 * 2 * 1)
def compute_probability(s, p):
    n = 256 * 256 * 256 / (3 * 2 * 1)
    max_error_ratio = 1 - s
    k = max_error_ratio * n
    k_int = int(round(k))
    
    mu = n * p
    variance = n * p * (1 - p)
    sigma = math.sqrt(variance)
    
    z_score = (k_int + 0.5 - mu) / sigma
    probability = norm.cdf(z_score)
    
    return probability

correct_fraction = 0.20
error_prob = 0.7998
probability = compute_probability(correct_fraction, error_prob)
print(-np.log(error_prob)/np.log(256**3/0.1))
print(f"≥{correct_fraction*100:.0f}%{error_prob*100:.0f}%，: {probability:.10f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit

x_labels = ["100%", "90%", "80%", "70%", "60%", "50%", "40%", "30%", "20%"]
x_values = np.arange(len(x_labels))
y_values = [100, 12.17, 8.5, 6.36, 4.84, 3.66, 2.70, 1.89, 1.18]

y_log_values = np.log(y_values)

def linear_func(x, m, c):
    return m * x + c

popt, pcov = curve_fit(linear_func, x_values, y_log_values)
m, c = popt

x_smooth = np.linspace(x_values.min(), x_values.max(), 300)
y_log_smooth = linear_func(x_smooth, m, c)

# --------------------------
# --------------------------
plt.figure(figsize=(6,4))
plt.plot(x_values, y_values, marker='o', color='#3c7fb1', linestyle='-', linewidth=2, label="Necessary Site")

for i, (x, y) in enumerate(zip(x_values, y_values)):
    plt.text(x, y, f'{y:.2f}', ha='center', va='bottom', fontsize=10)

ax = plt.gca()
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

plt.xlabel("Percentage of Correct Constraints", fontsize=12)
plt.ylabel("Necessary Site", fontsize=12)
plt.xticks(x_values, x_labels, rotation=30, ha='right')
plt.legend(fontsize=10)
plt.title("Necessary Site vs Correct Constraints", fontsize=14)
plt.tight_layout()
plt.show()

# --------------------------
# --------------------------
plt.figure(figsize=(5, 4))
plt.plot(x_values, y_log_values, marker='o', color='#3c7fb1', linestyle='-', linewidth=2, label="Log(Necessary Site)")

for i, (x, y) in enumerate(zip(x_values, y_log_values)):
    plt.text(x, y, f'{y:.2f}', ha='center', va='bottom', fontsize=10)

plt.plot(x_smooth, y_log_smooth, color='red', label="Linear Fit (Log Data)", linewidth=2)

ax = plt.gca()
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

plt.xlabel("Percentage of Correct Constraints", fontsize=12)
plt.ylabel("Log(Necessary Site)", fontsize=12)
plt.xticks(x_values, x_labels, rotation=30, ha='right')
plt.legend(fontsize=10)
plt.title("Log Data with Linear Fit", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
import scienceplots

plt.style.use(['science', 'no-latex'])

def solve_delta_star(d, lambda_param, q):
    return 0.6 * (np.exp(-lambda_param * d) * (1 - q) + q * np.exp(-lambda_param * (2 - d)))

error = 0.1
q = 0.04
l_star = 1 / 9
d = 1
lambda_param = 0.25

error_values = np.linspace(0.001, 0.2, 20)
k_values_error = []

for error in error_values:
    delta_star = minimize_scalar(solve_delta_star, bounds=(0, d), method='bounded', args=(lambda_param, q)).fun
    k1 = (-32 * np.log(error)) * q / (l_star ** 2 * delta_star**2)
    k2 = (-32 * np.log(error)) * (l_star + (1 - np.exp(-lambda_param)) * q) / (
        0.6 * lambda_param * l_star ** 2 * delta_star * (1 - q + q * np.exp(-2 * lambda_param)))
    k_values_error.append(np.maximum(k1, k2))

q_values = np.linspace(0.001, 0.3, 20)
k_values_q = []
error = 0.1
l_star = 1 / 9
d = 1
lambda_param = 0.25
for q in q_values:
    delta_star = minimize_scalar(solve_delta_star, bounds=(0, d), method='bounded', args=(lambda_param, q)).fun
    k1 = (-32 * np.log(error)) * q / (l_star ** 2 * delta_star**2)
    k2 = (-32 * np.log(error)) * (l_star + (1 - np.exp(-lambda_param)) * q) / (
        0.6 * lambda_param * l_star ** 2 * delta_star * (1 - q + q * np.exp(-2 * lambda_param)))
    k_values_q.append(np.maximum(k1, k2))

lambda_values = np.linspace(0.083, 5, 20)
k_values_lambda = []
error = 0.1
q = 0.04
l_star = 1 / 9
d = 1
for lambda_param in lambda_values:
    delta_star = minimize_scalar(solve_delta_star, bounds=(0, d), method='bounded', args=(lambda_param, q)).fun
    k1 = (-32 * np.log(error)) * q / (l_star ** 2 * delta_star**2)
    k2 = (-32 * np.log(error)) * (l_star + (1 - np.exp(-lambda_param)) * q) / (
        0.6 * lambda_param * l_star ** 2 * delta_star * (1 - q + q * np.exp(-2 * lambda_param)))
    k_values_lambda.append(np.maximum(k1, k2))

l_star_values = np.linspace(0.01, 0.3, 20)
k_values_l_star = []
error = 0.1
q = 0.04
d = 1
lambda_param = 0.25
for l_star in l_star_values:
    delta_star = minimize_scalar(solve_delta_star, bounds=(0, d), method='bounded', args=(lambda_param, q)).fun
    k1 = (-32 * np.log(error)) * q / (l_star ** 2 * delta_star**2)
    k2 = (-32 * np.log(error)) * (l_star + (1 - np.exp(-lambda_param)) * q) / (
        0.6 * lambda_param * l_star ** 2 * delta_star * (1 - q + q * np.exp(-2 * lambda_param)))
    k_values_l_star.append(np.maximum(k1, k2))

d_values = np.linspace(0.1, 1, 20)
k_values_d = []
error = 0.1
q = 0.04
l_star = 1 / 9
lambda_param = 0.25
for d in d_values:
    delta_star = minimize_scalar(solve_delta_star, bounds=(0, d), method='bounded', args=(lambda_param, q)).fun
    k1 = (-32 * np.log(error)) * q / (l_star ** 2 * delta_star**2)
    k2 = (-32 * np.log(error)) * (l_star + (1 - np.exp(-lambda_param)) * q) / (
        0.6 * lambda_param * l_star ** 2 * delta_star * (1 - q + q * np.exp(-2 * lambda_param)))
    k_values_d.append(np.maximum(k1, k2))

fig, axs = plt.subplots(2, 3, figsize=(14, 7))
axs = axs.flatten()

# Plot 1: error vs k
axs[0].plot(
    error_values,
    np.log10(k_values_error),
    color='tab:red',
    marker='o',
    linestyle='--',
    markerfacecolor='none',
    label=r"$\log_{10}(k)$ vs error"
)
axs[0].set_xlabel("Error of single triplet")
axs[0].set_ylabel(r"$\log_{10}$(Necessary $k$)")
axs[0].legend()

# Plot 2: q vs k
axs[1].plot(
    q_values,
    np.log10(k_values_q),
    color='tab:red',
    marker='o',
    linestyle='--',
    markerfacecolor='none',
    label=r"$\log_{10}(k)$ vs $q(ITO)$"
)
axs[1].plot(
    q_values,
    np.log10(np.array(k_values_q) * 0.1),
    color='gray',
    marker='<',
    linestyle='--',
    markerfacecolor='none',
    label=r"$\log_{10}(k)$ vs $q$(PTO)"
)
axs[1].set_xlabel("Mutation probability (q)")
axs[1].set_ylabel(r"$\log_{10}$(Necessary $k$)")
axs[1].legend()

# Plot 3: lambda vs k
axs[2].plot(
    lambda_values,
    np.log10(k_values_lambda),
    color='tab:red',
    marker='o',
    linestyle='--',
    markerfacecolor='none',
    label=r"$\log_{10}(k)$ vs $\lambda(ITO)$"
)
axs[2].plot(
    lambda_values,
    np.log10(np.array(k_values_lambda) * 0.1),
    color='gray',
    marker='<',
    linestyle='--',
    markerfacecolor='none',
    label=r"$\log_{10}(k)$ vs $\lambda(PTO)$"
)
axs[2].set_xlabel("Exponential decay parameter (lambda)")
axs[2].set_ylabel(r"$\log_{10}$(Necessary $k$)")
axs[2].legend()

# Plot 4: l_star vs k
axs[3].plot(
    l_star_values,
    np.log10(k_values_l_star),
    color='tab:red',
    marker='o',
    linestyle='--',
    markerfacecolor='none',
    label=r"$\log_{10}(k)$ vs $l^*(ITO)$"
)
axs[3].plot(
    l_star_values,
    np.log10(np.array(k_values_l_star) * 0.1),
    color='gray',
    marker='<',
    linestyle='--',
    markerfacecolor='none',
    label=r"$\log_{10}(k)$ vs $l^*(PTO)$"
)
axs[3].set_xlabel(r"$l^*$ (character length)")
axs[3].set_ylabel(r"$\log_{10}$(Necessary $k$)")
axs[3].legend()

# Plot 5: d vs k
axs[4].plot(
    d_values,
    np.log10(k_values_d),
    color='tab:red',
    marker='o',
    linestyle='--',
    markerfacecolor='none',
    label=r"$\log_{10}(k)$ vs $d(ITO)$"
)
axs[4].plot(
    d_values,
    np.log10(np.array(k_values_d) * 0.1),
    color='gray',
    marker='<',
    linestyle='--',
    markerfacecolor='none',
    label=r"$\log_{10}(k)$ vs $d(PTO)$"
)
axs[4].set_xlabel("Estimated depth (d)")
axs[4].set_ylabel(r"$\log_{10}$(Necessary $k$)")
axs[4].legend()

axs[5].plot(
    d_values,
    np.log10(k_values_d),
    color='tab:red',
    marker='o',
    linestyle='--',
    markerfacecolor='none',
    label=r"$\log_{10}(k)$ vs $d$ (PTO)"
)
axs[5].plot(
    d_values,
    np.log10(np.array(k_values_d) * 0.1),
    color='gray',
    marker='<',
    linestyle='--',
    markerfacecolor='none',
    label=r"$\log_{10}(k)$ vs $d (ITO)$"
)
axs[5].set_xlabel("Estimated depth (d)")
axs[5].set_ylabel(r"$\log_{10}$(Necessary $k$")
axs[5].legend()
for ax in axs:
    ax.grid(False)
plt.tight_layout()
plt.show()



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
import scienceplots

plt.style.use(['science', 'no-latex'])

def solve_delta_star(d, lambda_param, q):
    return 0.6 * (np.exp(-lambda_param * d) * (1 - q) + q * np.exp(-lambda_param * (2 - d)))

def compute_k(q, lambda_param, error=0.1, l_star=1/9, d=1):
    delta_star = minimize_scalar(solve_delta_star, bounds=(0, d), method='bounded', args=(lambda_param, q)).fun
    k1 = (-32 * np.log(error)) * q / (l_star ** 2 * delta_star**2)
    k2 = (-32 * np.log(error)) * (l_star + (1 - np.exp(-lambda_param)) * q) / (
        0.6 * lambda_param * l_star ** 2 * delta_star * (1 - q + q * np.exp(-2 * lambda_param)))
    return np.maximum(k1, k2)

q_values = np.linspace(0.001, 0.3, 9)
lambda_values = np.linspace(0.1, 5, 9)
Q, Lambda = np.meshgrid(q_values, lambda_values)
K = np.zeros_like(Q)

for i in range(Q.shape[0]):
    for j in range(Q.shape[1]):
        K[i, j] = compute_k(Q[i, j], Lambda[i, j])

plt.figure(figsize=(6, 4))
cp = plt.pcolormesh(Q, Lambda, np.log10(K), shading='auto', cmap='viridis')
plt.colorbar(cp, label=r'$\log_{10}(k)$')
plt.xlabel("Mutation probability (q)")
plt.ylabel(r"Exponential decay ($\lambda$)")
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import scipy.special as sp

def log_binom_pmf(k, n, p):
    """ log(P(X=k))， X~Bin(n, p)"""
    log_comb = sp.gammaln(n + 1) - sp.gammaln(k + 1) - sp.gammaln(n - k + 1)
    return log_comb + k * np.log(p) + (n - k) * np.log(1 - p)

def tail_prob_binomial_exact_logsum(n, c, s):
    """
     P(S > s) = P(X > sN)，X ~ Bin(N, 1-c)
     log 
    """
    N = sp.comb(n, 3, exact=True)
    p = 1 - c
    k_threshold = int(np.floor(s * N))  # P(X > sN) = sum_{k > sN} P(X=k)

    log_probs = [log_binom_pmf(k, N, p) for k in range(k_threshold, N + 1)]
    log_sum = sp.logsumexp(log_probs)
    return np.exp(log_sum)
prob = tail_prob_binomial_exact_logsum(n=256, c=0.13, s=0.9)
print(f"P(S > 0.9) = {prob:.6f}")

